### PHASE 5 — Friday | Crisis Timeline + Time Intelligence

---

`Business Request from CEO:`
"When did this crisis start? Was it sudden or gradual? I need Q4 2023 default trends for the board meeting next week."

`Mindset Unlock:`
The finale. Today's job is not just to count, but to predict. A 5% default rate that has stayed 5% for six months is stable. A 5% default rate that was 1% last month is a breakout. Velocity, the rate of change over time, is the most important metric for any executive. You are looking for the exact moment the EduFin portfolio lost its health.

`CEO'S EXECUTIVE BRIEF — "When did this start, and how fast is it moving?"` 

The Board of Directors is meeting on Monday. They need to know the 'Genesis Month' of this default surge and the current 'Velocity' (month-over-month growth rate). Your final report must identify the trend, show the acceleration, and synthesize insights from all previous phases into one coherent narrative.


### Start Spark Session

In [1]:
## Import dependencies and create Spark session
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 09:27:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Global Variables

In [26]:
N = 30 # Number of rows to show in results
schema = "edufin_small"  # edufin_small or edufin_national

### Load the data

In [27]:
## Load the identified datasets and create a temporary views for SQL queries
loans = spark.read.csv(f"../datasets/{schema}/loans.csv", header=True)
loans.createOrReplaceTempView("loans")
loans.show(n=10)

+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|loan_id|customer_id|institution_id|loan_amount|loan_status|interest_rate|loan_tenure_months|application_date|disbursement_date|maturity_date| emi_amount|     purpose_of_loan|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|      1|       2440|          2954| 190607.125|  Defaulted|  11.39000034|                84|      08-12-2021|       23-12-2021|   16-11-2028|3302.540039|     Living Expenses|
|      2|       2440|          4741| 425798.375|     Active|  14.43999958|                48|      01-01-2022|       11-01-2022|   21-12-2025|11728.73047|Course Fees + Living|
|      3|       2440|           902|  318341.25|  Defaulted|  11.64999962|                96|      06-03-2023|       12-

Query 5A (BRD): Monthly Disbursement Trends

---

- Calculate: Total loans and amount disbursed per month (last 24 months).
- Business Purpose: Visualize growth trajectory.

In [28]:
query = """
SELECT 
    MIN(disbursement_date) AS earliest_date,
    MAX(disbursement_date) AS latest_date,
    COUNT(*) AS total_records
FROM loans;
"""
spark.sql(query).show(truncate=False)

+-------------+-----------+-------------+
|earliest_date|latest_date|total_records|
+-------------+-----------+-------------+
|01-01-2022   |31-12-2024 |5000         |
+-------------+-----------+-------------+



In [ ]:
query = """
WITH monthly_disbursements AS (
    SELECT 
        DATE_FORMAT(
            COALESCE(
                try_to_date(disbursement_date, 'dd-MM-yyyy'),
                try_to_date(disbursement_date, 'yyyy-MM-dd')
            ), 
            'yyyy-MM'
        ) AS disbursement_month,
        COUNT(*) AS total_loans_disbursed,
        CASE 
            WHEN SUM(loan_amount) > 10000000.0 THEN CONCAT(ROUND(SUM(loan_amount) / 10000000.0, 2), ' Cr')
            ELSE CONCAT(ROUND(SUM(loan_amount) / 100000.0, 2), ' L') 
        END AS total_amount_disbursed
    FROM loans
    GROUP BY DATE_FORMAT(
        COALESCE(
            try_to_date(disbursement_date, 'dd-MM-yyyy'),
            try_to_date(disbursement_date, 'yyyy-MM-dd')
        ), 
        'yyyy-MM'
    )
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY disbursement_month DESC) AS SN,
    disbursement_month AS `Disbursement Month`,
    total_loans_disbursed AS `Loans Disbursed`,
    total_amount_disbursed AS `Amount Disbursed`
FROM monthly_disbursements
ORDER BY disbursement_month DESC;
"""
start = time.perf_counter()
spark.sql(query).show(n=100, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+---+------------------+---------------+----------------+
|SN |Disbursement Month|Loans Disbursed|Amount Disbursed|
+---+------------------+---------------+----------------+
|1  |2025-07           |2              |5.76 L          |
|2  |2025-06           |59             |2.71 Cr         |
|3  |2025-05           |119            |5.15 Cr         |
|4  |2025-04           |120            |4.87 Cr         |
|5  |2025-03           |98             |4.08 Cr         |
|6  |2025-02           |72             |2.83 Cr         |
|7  |2025-01           |137            |5.35 Cr         |
|8  |2024-12           |116            |4.61 Cr         |
|9  |2024-11           |100            |4.06 Cr         |
|10 |2024-10           |120            |4.95 Cr         |
|11 |2024-09           |100            |4.09 Cr         |
|12 |2024-08           |105            |4.59 Cr         |
|13 |2024-07           |124            |5.02 Cr         |
|14 |2024-06           |114            |4.89 Cr         |
|15 |2024-05  

STEP 5A (Workbook) — Open Metric: "Genesis Month" Discovery

---

What you're doing: The Board wants to know the EXACT moment we lost control. You must define what "Out of Control" means quantitatively (e.g., Velocity > 50%).

Your 'Genesis Month' Definition Logic:

*e.g., The month where (defaults / issuances) growth significantly deviates from the annual baseline. Propose your threshold.*

-> *Genesis Month = The first month where the default rate spiked by >50% compared to the previous month*

Your SQL Implementation:

In [31]:
query = """
WITH base_date AS (
    SELECT 
        EXTRACT(YEAR FROM COALESCE(
            try_to_date(disbursement_date, 'dd-MM-yyyy'),
            try_to_date(disbursement_date, 'yyyy-MM-dd')
        )) AS disbursement_year,
        CASE 
            WHEN EXTRACT(MONTH FROM COALESCE(
                try_to_date(disbursement_date, 'dd-MM-yyyy'),
                try_to_date(disbursement_date, 'yyyy-MM-dd')
            )) <= 3 THEN 'Q1'
            WHEN EXTRACT(MONTH FROM COALESCE(
                try_to_date(disbursement_date, 'dd-MM-yyyy'),
                try_to_date(disbursement_date, 'yyyy-MM-dd')
            )) <= 6 THEN 'Q2'
            WHEN EXTRACT(MONTH FROM COALESCE(
                try_to_date(disbursement_date, 'dd-MM-yyyy'),
                try_to_date(disbursement_date, 'yyyy-MM-dd')
            )) <= 9 THEN 'Q3'
            ELSE 'Q4'
        END AS quarter_name,
        CAST(loan_amount AS DECIMAL) AS loan_amount,
        loan_id,
        customer_id,
        loan_status
    FROM loans
),
quarterly_volume AS (
    SELECT 
        CONCAT(disbursement_year, '-', quarter_name) AS quarter,
        disbursement_year,
        quarter_name,
        ROUND(SUM(loan_amount) / 10000000, 2) AS amount_disbursed_cr,
        COUNT(loan_id) AS total_loans,
        COUNT(DISTINCT customer_id) AS total_customers,
        ROUND(AVG(loan_amount) / 100000, 2) AS avg_loan_size_lakh,
        ROUND(
            100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(loan_id),
            2
        ) AS default_rate_pct
    FROM base_date
    GROUP BY disbursement_year, quarter_name
),
portfolio_baseline AS (
    SELECT 
        ROUND(AVG(amount_disbursed_cr), 2) AS baseline_amount_cr,
        ROUND(AVG(total_loans), 0) AS baseline_loans,
        ROUND(AVG(default_rate_pct), 2) AS baseline_default_rate
    FROM quarterly_volume
),
velocity_detection AS (
    SELECT 
        qv.quarter,
        qv.disbursement_year,
        qv.quarter_name,
        qv.amount_disbursed_cr,
        qv.total_loans,
        qv.default_rate_pct,
        pb.baseline_default_rate,
        ROUND(
            100.0 * (qv.default_rate_pct - pb.baseline_default_rate) / pb.baseline_default_rate,
            2
        ) AS default_rate_velocity_pct,
        LAG(qv.default_rate_pct) OVER (ORDER BY qv.disbursement_year, qv.quarter_name) AS prev_default_rate,
        CASE 
            WHEN qv.default_rate_pct >= pb.baseline_default_rate + 2.0 THEN 'CRISIS'
            WHEN qv.default_rate_pct >= pb.baseline_default_rate + 1.0 THEN 'WARNING'
            ELSE 'NORMAL'
        END AS status
    FROM quarterly_volume qv
    CROSS JOIN portfolio_baseline pb
)
SELECT 
    quarter AS `Quarter`,
    total_loans AS `Loans Disbursed`,
    CONCAT(amount_disbursed_cr, ' Cr') AS `Amount Disbursed`,
    CONCAT(default_rate_pct, '%') AS `Default Rate (%)`,
    CONCAT(baseline_default_rate, '%') AS `Baseline Rate (%)`,
    CONCAT(default_rate_velocity_pct, '%') AS `Velocity (%)`,
    status AS `Status`,
    CASE 
        WHEN status IN ('CRISIS', 'WARNING') 
             AND (prev_default_rate IS NULL OR prev_default_rate < baseline_default_rate + 1.0)
        THEN '← GENESIS QUARTER'
        ELSE ''
    END AS genesis_marker
FROM velocity_detection
ORDER BY disbursement_year, quarter_name
"""
start = time.perf_counter()
spark.sql(query).show(n=100, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------+---------------+----------------+----------------+-----------------+------------+-------+-----------------+
|Quarter|Loans Disbursed|Amount Disbursed|Default Rate (%)|Baseline Rate (%)|Velocity (%)|Status |genesis_marker   |
+-------+---------------+----------------+----------------+-----------------+------------+-------+-----------------+
|2021-Q3|147            |5.93 Cr         |10.20%          |11.03%           |-7.52%      |NORMAL |                 |
|2021-Q4|320            |12.84 Cr        |10.31%          |11.03%           |-6.53%      |NORMAL |                 |
|2022-Q1|317            |12.97 Cr        |8.52%           |11.03%           |-22.76%     |NORMAL |                 |
|2022-Q2|306            |12.89 Cr        |10.46%          |11.03%           |-5.17%      |NORMAL |                 |
|2022-Q3|360            |14.61 Cr        |12.78%          |11.03%           |15.87%      |WARNING|← GENESIS QUARTER|
|2022-Q4|330            |13.34 Cr        |12.12%          |11.03

Query 5B (BRD): Tracking Defaults by Vintage

---

- Calculate: How many loans from each month's cohort eventually defaulted.
- Business Purpose: Cohort analysis to pinpoint bad vintage months.

In [ ]:
query = """
WITH vintage_cohorts AS (
    SELECT 
        date_format(disbursement_date, 'yyyy-MM') AS disbursement_vintage,
        COUNT(*) AS total_loans_in_vintage,
        SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaulted_loans,
        ROUND(100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct,
        ROUND(AVG(loan_amount), 2) AS avg_loan_amount
    FROM loans
    GROUP BY date_format(disbursement_date, 'yyyy-MM')
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY disbursement_vintage DESC) AS SN,
    disbursement_vintage AS `Vintage Month`,
    total_loans_in_vintage AS `Total Loans`,
    defaulted_loans AS `Defaulted Loans`,
    CONCAT(default_rate_pct, '%') AS `Default Rate (%)`
FROM vintage_cohorts
ORDER BY disbursement_vintage DESC;
"""

start = time.perf_counter()
result_df = spark.sql(query)
result_df.show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

STEP 5B (Workbook) — Monthly Default Trend

---

What you're doing: Count how many loans defaulted each month. Look for the first month where defaults exceeded 50, then 100, then 200. This is the raw timeline of the crisis.

> When did the count first double in a single month? That is your 'Trigger Point'.

In [ ]:
query = """
WITH monthly_defaults AS (
    SELECT 
        date_format(default_date, 'yyyy-MM') AS default_month,
        COUNT(*) AS defaulted_count
    FROM loans
    WHERE loan_status = 'Defaulted' 
      AND default_date IS NOT NULL
    GROUP BY 1
),
trend_analysis AS (
    SELECT 
        default_month,
        defaulted_count,
        LAG(defaulted_count) OVER (ORDER BY default_month) AS previous_month_count
    FROM monthly_defaults
)
SELECT 
    default_month AS `Calendar Month`,
    defaulted_count AS `Current Month Default`,
    previous_month_count AS `Previous Month Default`,
    CONCAT(ROUND(((defaulted_count - previous_month_count) * 100.0 / previous_month_count), 2), '%') AS `Growth (%)`
FROM trend_analysis
ORDER BY default_month;
"""

start = time.perf_counter()
result_df = spark.sql(query)
result_df.show(n=50, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

Query 5C (BRD): Monthly Default Rate Trends

---

- Calculate: Default Rate % for each month.
- Business Purpose: The core "Crisis Timeline" metric.

STEP 5C (Workbook) — Default Velocity (Month-over-Month Growth)

---

What you're doing: Use LAG() or a self-join to compare this month's defaults to last month's defaults. Calculate the percentage growth. This 'Velocity' metric is what captures executive attention.

> Is the velocity increasing, decreasing, or plateauing in the most recent two months? This determines if the crisis is 'out of control' or 'stabilizing'.

Query 5D (BRD): Cumulative Financial Impact

---

- Calculate: Rolling total of losses over time.
- Business Purpose: See how the financial hole grew.

STEP 5D (Workbook) — Lag Time (Origination to Default)

---

What you're doing: Calculate the average number of days between the loan start date and the default date. This shows 'Survival Time'. If survival time is dropping, it means new loans are defaulting faster than old ones.

> Note the lowest average survival time. (e.g., 'In August, loans were defaulting in just X days on average').

Query 5E (BRD): Crisis Timeline Dashboard

---

- Calculate: Month-by-month crisis heat map.
- Business Purpose: Pinpoint the exact "Point of Failure" (month where defaults spiked).
- Classifications: CRISIS MONTH (>15%), WARNING MONTH (10-15%).
- REQUIRED Text Summary: Identify the exact month the crisis began and whether it is currently stabilizing or worsening.

STEP 5E (Workbook) — Final Strategic Synthesis (Executive Dashboard)

---

What you're doing: Combine your core findings from all phases into a single, high-stakes table. This is the summary the Board of Directors will use to decide if they should fire the CEO or double down on collections.


> Read your 'Analyst Notes' (status labels). Do they tell a clear story of deterioration across the phases? 